# Optuna tuning

Load search-space tags from a YAML file and resolve trial-specific Julia configs.


In [ ]:
function find_repo_root(start = pwd())
    dir = abspath(start)
    while !isfile(joinpath(dir, "Project.toml")) || !isdir(joinpath(dir, "src"))
        parent = dirname(dir)
        parent == dir && error("Could not find QuantumGraph.jl repository root")
        dir = parent
    end
    dir
end

repo_root = find_repo_root()
cd(repo_root)
using Pkg
Pkg.activate(repo_root)


In [ ]:
using QuantumGraph
using Optuna

search_file = load_config(read(joinpath(repo_root, "docs", "src", "examples", "configs", "optuna_search_space.yml"), String))
search_config = Dict{String, Any}(
    "model" => Dict{String, Any}(
        "layers" => search_file["layers"],
        "activation" => search_file["activation"],
        "width" => Dict("type" => "coupled-sweep", "target" => ["model", "layers"], "values" => [16, 32, 64]),
        "learning_rate" => search_file["learning_rate"],
    ),
    "trainer" => Dict{String, Any}(
        "epochs" => search_file["epochs"],
        "copied_layers" => Dict("type" => "reference", "target" => ["model", "layers"]),
    ),
)


In [ ]:
trial = FixedTrial(Dict{String, Any}(
    "model.layers" => 2,
    "model.activation" => "relu",
    "model.learning_rate" => 0.001,
    "trainer.epochs" => 2,
))
resolved = build_search_space(search_config, trial)
resolved


In [ ]:
study = create_tuning_study(Dict("study_name" => "quantumgraph-demo", "direction" => "minimize", "storage" => nothing))
study.study_name, study.direction, study.backend


In [ ]:
best_config_path = joinpath(mktempdir(), "best_config.txt")
best_config = save_best_config(search_config, trial, best_config_path)
best_config, best_config_path
